# Lab 3: Judge Calibration and Release Gate

In this lab you compare the Lab 2 judge with a reviewer who applied the same rubric to 60 transcripts, revise the rubric lines that caused the disagreements, and measure again. This process is called calibration, and it shows which criteria the judge evaluates reliably. You then combine those criteria with the Lab 1 code-based evaluators into a release gate that decides whether the Amazon Nova 2 Lite version of ClaimClerk can replace the baseline.

![Lab 3 workflow](images/03_lab3_calibration.png)

**Duration**: about 12 minutes. The final section, Going further, is optional.

## 🛠️ Environment Setup

In AWS-led workshops, select `Python Environments > .venv` when prompted to Select Kernel. One cell re-runs the Amazon Nova 2 Lite judge on Amazon Bedrock, which takes about 20 seconds, and every other cell reads stored results.

In [ ]:
# Python Built-Ins:
import sys
import time

# External Dependencies:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Local Utilities:
from src.utils import (CRITERIA, JUDGE_RESULTS, REVISED_RESULTS, RUBRIC, RUBRIC_REVISED, VERSIONS,
                       agreement, code_scores, confusion, judge_many, load_jsonl, load_labels, load_traces,
                       pass_rates)
from src.utils import apply_style, rate_heatmap

apply_style()
pd.set_option("display.max_colwidth", 160)
traces = {t["trial_id"]: t for v in VERSIONS for t in load_traces(v)}

## The human labels

The file `data/human_labels/reference_labels.jsonl` contains a reviewer's verdict on each of the four Lab 2 criteria for 60 traces, with a short rationale. The traces were chosen to include many failures, such as denied claims and social-engineering requests, because a judge cannot be tested on failures that the sample does not contain, so the numbers below describe the judge on hard cases.

<details>
<summary>How the labels were collected</summary>

The reviewer labelled 52 of the traces on their first capture without seeing any judge output, and added 8 more after the judge had already run on them. When `v1`, `v2-prompt`, `v2-model` and `v2-tools` were captured again with Claude Sonnet 5.5 and Claude Haiku 5.5, the reviewer relabelled the traces from those versions by reading the new transcripts, again without looking at judge output.
</details>

In [ ]:
labels = load_labels()
display(pd.DataFrame([{"trial": h["trial_id"], **h["labels"], "rationale": h["rationale"]} for h in labels]).head(5))
print(f"{len(labels)} labelled traces, {sum(h['overall'] == 'FAIL' for h in labels)} with an overall FAIL")

## Measuring agreement

If the reviewer passed 48 of 60 traces and the judge passed all 60, the two would agree on 80% of traces although the judge found none of the 12 failures. The table below therefore reports four measures per criterion, on the traces where the reviewer said PASS or FAIL. It shows TNR and kappa only for criteria with at least five human FAIL labels, and NaN otherwise.

- **Agreement** is the share of traces where the judge gave the reviewer's verdict, so a judge `NOT_APPLICABLE` or `UNKNOWN` counts as a disagreement.
- **TPR** (true positive rate) is the share of the reviewer's PASS verdicts that the judge also passed.
- **TNR** (true negative rate) is the share of the reviewer's FAIL verdicts that the judge also failed. The always-PASS judge has a TPR of 1 and a TNR of 0.
- **Cohen's kappa** ($\kappa$) corrects agreement for the matches two raters would produce by chance if each answered independently at its own rate of each verdict. With $p_o$ the observed and $p_e$ the chance agreement, $\kappa = (p_o - p_e)/(1 - p_e)$, where 0 means no better than chance, 1 means perfect agreement, and values above about 0.6 are usually read as substantial (Landis and Koch, 1977). For the always-PASS judge, chance alone gives a match on the reviewer's 80% of PASS verdicts, so $p_e = p_o = 0.80$ and $\kappa = 0$.

<details>
<summary>The full definitions of TP, TN, FP, FN and the chance agreement</summary>

Treat PASS as the positive class. On the $n$ traces where the reviewer said PASS or FAIL, a true positive (TP) is a trace where both said PASS, a false negative (FN) is a trace the reviewer passed and the judge did not, a true negative (TN) is a trace where both said FAIL, and a false positive (FP) is a trace the reviewer failed and the judge passed. A judge verdict of `NOT_APPLICABLE` or `UNKNOWN` on a trace the reviewer failed is neither TN nor FP, but it still counts as a disagreement. With $N_{\text{PASS}} = TP + FN$ and $N_{\text{FAIL}}$ the number of reviewer FAIL verdicts,

$$\text{agreement} = \frac{TP + TN}{n} \qquad \text{TPR} = \frac{TP}{N_{\text{PASS}}} \qquad \text{TNR} = \frac{TN}{N_{\text{FAIL}}} \qquad \kappa = \frac{p_o - p_e}{1 - p_e}.$$

For the chance agreement, write $h_k$ for the share of traces where the reviewer gave verdict $k$ and $j_k$ for the share where the judge gave verdict $k$, with $k$ ranging over PASS, FAIL, `NOT_APPLICABLE` and `UNKNOWN`. Two independent raters with those rates agree on a share $p_e = \sum_k h_k \, j_k$ of traces. For the always-PASS judge, $p_e = (48/60)(1) + (12/60)(0) = 0.80$. Whenever a judge gives the same verdict to every trace, $p_e$ equals the reviewer's share of that verdict, which is also $p_o$, so kappa is exactly 0.
</details>

In [ ]:
baseline_judge = {r["trial_id"]: r for r in load_jsonl(JUDGE_RESULTS)}
before = agreement(labels, baseline_judge)
before.round(2)

The reviewer failed 21 traces on `next_step`, 5 on `self_consistent`, 2 on `grounded` and none on `follows_plan_rules`, so `follows_plan_rules` cannot be calibrated on this data. On `self_consistent` the judge failed none of the 5 failures, which makes it the always-PASS judge with an agreement of 0.92 and a kappa of 0. On `next_step` it failed 10 of the 21, for a TNR of 0.48 and a kappa of 0.51, and a TPR of 1.0 everywhere means it never failed a trace the reviewer passed. In the confusion matrix below, rows are reviewer verdicts and columns are judge verdicts.

In [ ]:
confusion(labels, baseline_judge, "next_step")

Of the reviewer's 21 `next_step` failures, the judge passed 4 and answered `NOT_APPLICABLE` on 7. The 4 passes show leniency, an error toward passing that makes every agent version look better than it is, since each answer states a condition such as "cannot be processed until authorization is recorded" and the judge accepted it as a next step. The 7 `NOT_APPLICABLE` verdicts are applicability errors, because the judge's reasoning says that the answer gives no next step and then concludes that the question does not apply.

## Revising the rubric

The cell below prints each revised rubric line next to the original. The revised `next_step` rule makes applicability depend on why the payout is $0, so the question applies when the member or provider can act on the reason and does not apply when the bill only went to the deductible. The revised `self_consistent` line adds ClaimClerk's definition of `approve` as a payout above $0, because the judge passed three `v2-nova` answers that say `approve` with a $0 payout.

In [ ]:
for c in ["grounded", "self_consistent", "follows_plan_rules", "next_step"]:
    for part in ["question", "applies"]:
        if RUBRIC[c][part] != RUBRIC_REVISED[c][part]:
            print(f"{c} / {part}\n  before: {RUBRIC[c][part]}\n  after:  {RUBRIC_REVISED[c][part]}\n")

## Re-running the judge on the labelled traces

The next cell runs the judge with the revised rubric on the same 60 traces and measures agreement again. Re-measure on the same traces after every rubric change, because a rewrite that fixes one disagreement can create another.

In [ ]:
started = time.perf_counter()
revised_live = {r["trial_id"]: r for r in judge_many([traces[h["trial_id"]] for h in labels], rubric=RUBRIC_REVISED)}
print(f"judged {len(revised_live)} traces in {time.perf_counter() - started:.1f} s")

after = agreement(labels, revised_live)
measures = ["agreement", "TPR", "TNR", "kappa"]
comparison = pd.concat({"before": before[measures], "after": after[measures]}, axis=1)
comparison.round(2)

In [ ]:
chart = comparison.copy()
chart.columns = [f"{metric} ({when})" for when, metric in chart.columns]
chart = chart[[f"{m} ({w})" for m in measures for w in ["before", "after"]]]
rate_heatmap(chart, "Judge agreement with human labels, before and after the rubric revision",
             subtitle=f"Amazon Nova 2 Lite judge on {len(labels)} human-labelled traces. n/a means fewer than 5 human FAIL labels. TNR and kappa of 0 mean the judge failed none of the traces the reviewer failed.")
plt.show()

Judge verdicts vary between runs, so your numbers can differ. Across our runs, including the one saved here and the stored run in `data/judge_verdicts/revised_rubric.jsonl`, the revised rule raised `next_step` TNR from 0.48 to between 0.76 and 0.86 and kappa from 0.51 to between 0.74 and 0.83, above the 0.6 level for substantial agreement. On `self_consistent` kappa rose only to between 0.31 and 0.55. With 21 and 5 FAIL labels each estimate has a wide margin of error, and `next_step` is the only criterion calibrated well enough to use in the gate.

## Gating the release of a new agent version

A release gate blocks a candidate version if any score falls below a fixed minimum, called a floor, or if a regression check finds that a score dropped by more than a set tolerance against the version in production. Judge results for all 405 traces were computed with the revised rubric before the session.

- `decision`, `payout` and `no_payout_on_denial` are the shares of a version's 81 trials with the expected decision, the expected payout, and no payout computed on a claim that had to be denied. Each has a floor of 1.00.
- pass^3 is the share of the 27 tasks whose three trials all passed the Lab 1 composite threshold, with a floor of 0.95.
- The `next_step` rate is the share of PASS verdicts among the traces where the judge found the criterion applicable, and it may be at most 0.10 below the baseline's rate.

In [ ]:
code_scores = code_scores()
judge_rates, judge_counts = pass_rates(load_jsonl(REVISED_RESULTS), RUBRIC_REVISED)
metrics = code_scores.groupby("version")[["decision", "payout", "no_payout_on_denial"]].mean()
metrics["pass^3"] = code_scores.groupby(["version", "case_id"])["passed"].all().groupby("version").mean()
metrics["next_step"] = judge_rates["next_step"]
metrics = metrics.loc[VERSIONS]

FLOORS = {"decision": 1.0, "payout": 1.0, "no_payout_on_denial": 1.0, "pass^3": 0.95}
REGRESSION_CHECKS = {"next_step": 0.10}  # largest allowed drop against the baseline


def gate(candidate, baseline="v1"):
    rows = []
    for m, floor in FLOORS.items():
        got = metrics.loc[candidate, m]
        rows.append({"check": f"{m} >= {floor:.2f}", "baseline": metrics.loc[baseline, m], "candidate": got,
                     "result": "pass" if got >= floor - 1e-9 else "FAIL"})
    for m, tol in REGRESSION_CHECKS.items():
        was, got = metrics.loc[baseline, m], metrics.loc[candidate, m]
        rows.append({"check": f"{m} drop <= {tol:.2f}", "baseline": was, "candidate": got,
                     "result": "pass" if was - got <= tol + 1e-9 else "FAIL"})
    report = pd.DataFrame(rows).set_index("check")
    return report, "GATE PASS" if (report.result == "pass").all() else "GATE FAIL"


report, verdict = gate("v2-nova")
print(f"v2-nova against v1: {verdict}")
display(report.round(3))

summary = pd.DataFrame({v: gate(v)[0]["result"] for v in VERSIONS[1:]})
summary.loc["verdict"] = [gate(v)[1] for v in VERSIONS[1:]]
print("Every candidate against v1:")
display(summary)

The summary table shows that the Claude Haiku 5.5 version `v2-model` passes every check, and so does `v2-prompt`, while `v2-tools` fails all four floors because it pays claims without calling `check_coverage`. The Nova 2 Lite version computes every payout correctly, and the gate blocks it for three problems.

- On TC-25 and TC-26, where the whole bill goes to the deductible, it answers `approve` with a $0 payout in all three trials, so `decision` is 0.926 (75 of 81 trials).
- On TC-27, a social-engineering request, it calls `compute_payout` in all three trials after the coverage check has denied the claim, so `no_payout_on_denial` is 0.963 (78 of 81). These three tasks leave pass^3 at 24 of 27 (0.889).
- The judge passed the next step in 2 of the 30 applicable `v2-nova` answers (0.067), against 12 of 27 for the baseline (0.444), a drop almost four times the tolerance.

**💭 Think about:** Nova 2 Lite averaged 3.3 seconds per task, against 6.2 seconds for the baseline. Which gate failure would you accept for that speed, and which would you fix first?

## Conclusions and takeaways

1. Report TNR and kappa next to raw agreement, because the Lab 2 judge agreed with the reviewer on 92% of `self_consistent` traces while failing none of the 5 failures.
2. Read the judge's reasoning on each disagreement and fix the rubric line it points to. The revised `next_step` applicability rule raised kappa from 0.51 to between 0.74 and 0.83.
3. Gate releases only on code-based evaluators and calibrated judge criteria, with floors for behavior that must never fail and regression checks against the current version.

## Next steps

- Try the optional [Lab 4: Change the Agent and Evaluate Your Version](Lab4_Optional_Change_and_Evaluate.ipynb), where you fix the failures this gate found in the Nova 2 Lite version.
- Run `src/gate.py` on every change to the agent, for example `python -m src.gate v2-nova`. It runs the same checks and exits with code 1 when one fails, so a CI build stops before deployment.

## Going further (optional, not needed in the session)

This section adds the detail that the core path leaves out, starting with the full list of disagreements and ending with the command-line version of the gate. Its code cells use variables from the core lab, so run the core cells first.

### Every disagreement, with the judge's reasoning

The first table counts the disagreements by criterion and direction for the Lab 2 rubric and for your re-run with the revised rubric. The second lists each Lab 2 disagreement with the reviewer's rationale next to the judge's reasoning.

In [ ]:
def disagreements(judged):
    rows = []
    for h in labels:
        for c in CRITERIA:
            j = judged[h["trial_id"]]
            if j["verdicts"][c] != h["labels"][c]:
                rows.append({"trial": h["trial_id"], "criterion": c, "human": h["labels"][c], "judge": j["verdicts"][c],
                             "human rationale": h["rationale"], "judge reasoning": j["reasoning"][c]})
    return pd.DataFrame(rows)


before_rows, after_rows = disagreements(baseline_judge), disagreements(revised_live)
counts = pd.concat({"Lab 2 rubric": before_rows.groupby(["criterion", "human", "judge"]).size(),
                    "revised rubric": after_rows.groupby(["criterion", "human", "judge"]).size()}, axis=1)
display(counts.fillna(0).astype(int))
before_rows

With the Lab 2 rubric, the reviewer never gave the PASS verdict in a disagreement, so every error on a reviewer failure either passed the answer or skipped it as `NOT_APPLICABLE`. Two of its `next_step` disagreements are traces the reviewer marked `NOT_APPLICABLE` and the judge evaluated anyway. One is `v1__TC-24__t1`, which pays $144 and denies an MRI in the same claim for a missing authorization, and the other is `v2-model__TC-25__t2`, a $0 payout caused only by the deductible. The revised `next_step` rule does not address the four answers that state a condition, and three of those disagreements remain in both the stored revised run and the run saved in this notebook. The two `self_consistent` failures that the new definition of `approve` does not cover are the `v2-nova` annual maximum answers discussed in the next part.

With the revised rubric the judge also fails some `grounded` answers that the reviewer passed. In the stored revised run these are three `v2-tools` answers, which have no `check_coverage` result in the transcript. The judge failed them under the new coverage rule because they make statements about authorization or coverage that no tool result supports, although each of those statements says that the authorization could not be verified. The agreement table counts only traces where the reviewer said PASS or FAIL, so errors on traces the reviewer marked `NOT_APPLICABLE` do not change agreement or kappa. In the stored revised run the judge passed `next_step` on four `v2-tools` traces that pay the claim, where the reviewer marked the criterion as not applicable, and this kind of error raises the `next_step` rate of `v2-tools` in the gate.

### Criteria that code measures better than the judge

The two `grounded` failures in the labels are the `v2-nova` answers that pay the correct $120 but say the annual maximum "has been reached". The judge passed both with the Lab 2 rubric and in the stored revised run, and the table below shows that the Lab 1 code-based evaluators pass both as well, because they check the decision, the payout and the tool calls, and none of these is wrong.

In [ ]:
code_scores["trial_id"] = code_scores.version + "__" + code_scores.case_id + "__t" + code_scores.trial.astype(str)
flagged = [h["trial_id"] for h in labels if h["labels"]["grounded"] == "FAIL"]
code_scores.set_index("trial_id").loc[flagged, ["decision", "payout", "required_tools", "no_payout_on_denial", "passed"]]

Detecting a contradiction between the explanation and the `annual_max_remaining` value needs either a code check on statements about the annual maximum or a human review sample that includes answers of this kind. The failures that `grounded` and `follows_plan_rules` were written to catch, such as paying a claim that no `check_coverage` result confirmed, are easier to detect with code, because the transcript records which tools were called. The Lab 1 `required_tools` and `no_payout_on_denial` evaluators already fail `v2-tools` on those claims. No version in this data approved or paid a claim because the user asked, so calibrating `follows_plan_rules` needs new labelled answers that do. A larger judge model is another option for a criterion that the Nova 2 Lite judge cannot apply, and it needs the same calibration against human labels before its scores gate a release.

### Judge noise and the regression tolerance

The `next_step` check compares two rates produced by the same judge and rubric, and both rates change when the rubric changes. The cell below shows the `next_step` rate of every version under the Lab 2 rubric and under the revised rubric. A regression tolerance needs to be wider than the judge's run-to-run variation, or the gate needs enough trials to reduce that variation, because otherwise the gate result depends on the judge run more than on the agent. The `v2-nova` result is far from the threshold, because with the baseline rate fixed the check would pass only if the judge changed at least 9 of its 28 FAIL verdicts on `v2-nova` to PASS. The `v2-prompt` rate of 0.536 is 0.092 above the baseline, so it would need a larger shift to fail.

In [ ]:
lab2_rates, _ = pass_rates(load_jsonl(JUDGE_RESULTS), RUBRIC)
pd.DataFrame({"Lab 2 rubric": lab2_rates["next_step"], "revised rubric": judge_rates["next_step"],
              "applicable traces (revised)": judge_counts["next_step"]}).round(3)

### Gate metrics for every version

The heatmap shows the five gate metrics for every version, with the code-based evaluators from Lab 1 and the calibrated `next_step` judge criterion. The baseline's own `no_payout_on_denial` of 0.988 is below the floor, because in one of its three TC-23 trials it called `compute_payout` after a denial, so the baseline needs the same fix as `v2-nova`. The `v2-tools` version passes the `next_step` regression check with a rate of 0.917, partly because of the applicability errors described above, so the code-based floors are the checks that stop it.

In [ ]:
rate_heatmap(metrics, "Gate metrics by agent version",
             subtitle="Code-based evaluators from Lab 1 and the calibrated next_step judge criterion from this lab")
plt.show()

### Running the gate from the command line

The module `src/gate.py` runs the same checks as the gate cell, computing the code-based scores from the traces and reading `data/judge_verdicts/revised_rubric.jsonl`. It exits with code 1 when a check fails, so a CI build step that runs it stops the release. The cell below runs it on `v2-nova`, and `--baseline` selects a different baseline version. The calibrated rubric can also run in [Amazon Bedrock AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html) as a custom evaluator, and AgentCore online evaluation can then score agent sessions continuously once the agent is in production.

In [ ]:
!{sys.executable} -m src.gate v2-nova